# 03 Anomaly Detection
## Nifty Financial Platform

This notebook uses machine learning to identify anomalous financial behavior. We use the Isolation Forest algorithm to detect outliers that deviate significantly from the norm.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sqlalchemy import create_engine
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from dotenv import load_dotenv

# Settings
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

# Load environment variables
load_dotenv(dotenv_path='../.env')
DATABASE_URL = os.getenv("DATABASE_URL")

if not DATABASE_URL:
    print("ERROR: DATABASE_URL not found in .env file")
else:
    engine = create_engine(DATABASE_URL)
    print("Database engine created successfully.")

## 1. Prepare Data for Anomaly Detection
We select key ratios that are most likely to show anomalies.

In [ ]:
def fetch_anomaly_data(engine):
    query = """
    SELECT 
        dc.symbol, dc.company_name, ds.sector_name, 
        pl.net_profit_margin_pct, 
        bs.debt_to_equity,
        an.stock_pe, an.roce_pct
    FROM fact_profit_loss pl
    JOIN dim_company dc ON pl.company_id = dc.company_id
    JOIN dim_sector ds ON dc.sector_id = ds.sector_id
    JOIN fact_balance_sheet bs ON pl.company_id = bs.company_id AND pl.year_id = bs.year_id
    JOIN fact_analysis an ON pl.company_id = an.company_id AND pl.year_id = an.year_id
    """
    return pd.read_sql(query, engine)

df = fetch_anomaly_data(engine).dropna()
features = ['net_profit_margin_pct', 'debt_to_equity', 'stock_pe', 'roce_pct']
df.head()

## 2. Train Isolation Forest
Isolation Forest is an unsupervised algorithm that detects anomalies by isolating observations.

In [ ]:
# Scale the data
scaler = StandardScaler()
X = scaler.fit_transform(df[features])

# Fit Isolation Forest
# contamination: expected proportion of outliers
iso_forest = IsolationForest(n_estimators=100, contamination=0.05, random_state=42)
df['anomaly_score'] = iso_forest.fit_predict(X)

# -1 indicates an anomaly, 1 indicates normal
df['is_anomaly'] = df['anomaly_score'].apply(lambda x: 'Anomaly' if x == -1 else 'Normal')

print(f"Detected {len(df[df['anomaly_score'] == -1])} anomalies out of {len(df)} records.")

## 3. Visualize Anomalies

In [ ]:
sns.scatterplot(data=df, x='net_profit_margin_pct', y='debt_to_equity', hue='is_anomaly', palette={'Normal': 'blue', 'Anomaly': 'red'}, alpha=0.6)
plt.title('Anomaly Detection: Profit Margin vs Debt/Equity')
plt.show()

In [ ]:
sns.boxplot(data=df, x='is_anomaly', y='stock_pe', palette='Set2')
plt.title('Stock P/E Distribution: Normal vs Anomaly')
plt.show()

## 4. Deep Dive into Anomalous Companies

In [ ]:
anomalies = df[df['is_anomaly'] == 'Anomaly'].sort_values('net_profit_margin_pct', ascending=False)
print("Top Anomalous Records identified:")
display(anomalies.head(10))

## 5. Export Anomalies
Saving the detected anomalies for further investigation by financial analysts.

In [ ]:
os.makedirs('../data/analysis', exist_ok=True)
anomalies.to_csv('../data/analysis/detected_anomalies.csv', index=False)
print("Anomalies exported to data/analysis/detected_anomalies.csv")